In [2]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [3]:
# 데이터 로드
df_portfolio = pd.read_csv('/Users/wonwoo_mac/Desktop/project3/data/portfolio.csv')
df_profile = pd.read_csv('/Users/wonwoo_mac/Desktop/project3/data/profile.csv')
df_menu = pd.read_csv('/Users/wonwoo_mac/Desktop/project3/data/starbucks_menu_260112.csv')
df_transcript = pd.read_csv('/Users/wonwoo_mac/Desktop/project3/data/transcript.csv')

print("="*60)
print("데이터 로드 완료!")
print("="*60)
print(f"\nPortfolio: {df_portfolio.shape}")
print(f"Profile: {df_profile.shape}")
print(f"Menu: {df_menu.shape}")
print(f"Transcript: {df_transcript.shape}")

데이터 로드 완료!

Portfolio: (10, 7)
Profile: (17000, 6)
Menu: (195, 13)
Transcript: (306534, 5)


In [4]:
print("="*60)
print("데이터셋 정보:")
print("="*60)
print(df_portfolio.info())
print(df_profile.info())
print(df_transcript.info())
print(df_menu.info())

데이터셋 정보:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  10 non-null     int64
 1   reward      10 non-null     int64
 2   channels    10 non-null     str  
 3   difficulty  10 non-null     int64
 4   duration    10 non-null     int64
 5   offer_type  10 non-null     str  
 6   id          10 non-null     str  
dtypes: int64(4), str(3)
memory usage: 692.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        17000 non-null  int64  
 1   gender            14825 non-null  str    
 2   age               17000 non-null  int64  
 3   id                17000 non-null  str    
 4   became_member_on  17000 non-null  int64  
 5   income            14825 non-null  float64
dtypes: float64(1), i

In [ ]:
print("="*60)
print("데이터셋 샘플:")
print("="*60)
print(df_portfolio.head())
print(df_profile.head())
print(df_menu.head())

데이터셋 샘플:
   Unnamed: 0  reward                              channels  difficulty  \
0           0      10         ['email', 'mobile', 'social']          10   
1           1      10  ['web', 'email', 'mobile', 'social']          10   
2           2       0            ['web', 'email', 'mobile']           0   
3           3       5            ['web', 'email', 'mobile']           5   
4           4       5                      ['web', 'email']          20   

   duration     offer_type                                id  
0         7           bogo  ae264e3637204a6fb9bb56bc8210ddfd  
1         5           bogo  4d5c57ea9a6940dd891ad53e9dbe8da0  
2         4  informational  3f207df678b143eea3cee63160fa8bed  
3         7           bogo  9b98b8c7a33c4b65b9aebfe6a799e6d9  
4        10       discount  0b1e1539f2cc45b7b9fa7c272da2e1d7  
   Unnamed: 0 gender  age                                id  became_member_on  \
0           0    NaN  118  68be06ca386d4c31939f3a4f0e3dd783          20170212   


In [30]:
print(df_transcript.head(10))

   Unnamed: 0                            person           event  \
0           0  78afa995795e4d85b5d9ceeca43f5fef  offer received   
1           1  a03223e636434f42ac4c3df47e8bac43  offer received   
2           2  e2127556f4f64592b11af22de27a7932  offer received   
3           3  8ec6ce2a7e7949b1bf142def7d0e0586  offer received   
4           4  68617ca6246f4fbc85e91a2a49552598  offer received   
5           5  389bc3fa690240e798340f5a15918d5c  offer received   
6           6  c4863c7985cf408faee930f111475da3  offer received   
7           7  2eeac8d8feae4a8cad5a6af0499a211d  offer received   
8           8  aa4862eba776480b8bb9c68455b8c2e1  offer received   
9           9  31dda685af34476cad5bc968bdb01c53  offer received   

                                              value  time  
0  {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}     0  
1  {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}     0  
2  {'offer id': '2906b810c7d4411798c6938adc9daaa5'}     0  
3  {'offer id': 'fafdc

In [6]:
print("="*60)
print("데이터셋 기초 통계:")
print("="*60)
display(df_portfolio.describe(include='all').T)
display(df_profile.describe(include='all').T)
display(df_transcript.describe(include='all').T)
display(df_menu.describe(include='all').T)

데이터셋 기초 통계:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,10.0,NaN,NaN,NaN,4.5,3.02765,0.0,2.25,4.5,6.75,9.0
reward,10.0,NaN,NaN,NaN,4.2,3.583915,0.0,2.0,4.0,5.0,10.0
channels,10,4,"['web', 'email', 'mobile', 'social']",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
difficulty,10.0,NaN,NaN,NaN,7.7,5.831905,0.0,5.0,8.5,10.0,20.0
duration,10.0,NaN,NaN,NaN,6.5,2.321398,3.0,5.0,7.0,7.0,10.0
offer_type,10,3,bogo,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id,10,10,ae264e3637204a6fb9bb56bc8210ddfd,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,17000.0,NaN,NaN,NaN,8499.5,4907.621624,0.0,4249.75,8499.5,12749.25,16999.0
gender,14825,3,M,8484,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,17000.0,NaN,NaN,NaN,62.531412,26.73858,18.0,45.0,58.0,73.0,118.0
id,17000,17000,68be06ca386d4c31939f3a4f0e3dd783,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
became_member_on,17000.0,NaN,NaN,NaN,20167034.234118,11677.499961,20130729.0,20160526.0,20170802.0,20171230.0,20180726.0
income,14825.0,NaN,NaN,NaN,65404.991568,21598.29941,30000.0,49000.0,64000.0,80000.0,120000.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,306534.0,NaN,NaN,NaN,153266.5,88488.888045,0.0,76633.25,153266.5,229899.75,306533.0
person,306534,17000,94de646f7b6041228ca7dec82adb97d2,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event,306534,4,transaction,138953,NaN,NaN,NaN,NaN,NaN,NaN,NaN
value,306534,5121,{'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},14983,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,306534.0,NaN,NaN,NaN,366.38294,200.326314,0.0,186.0,408.0,528.0,714.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,195.0,NaN,NaN,NaN,97.0,56.435804,0.0,48.5,97.0,145.5,194.0
제품코드,195.0,NaN,NaN,NaN,7061368479720.610352,3764908927473.331055,2.0,4004000000077.5,9200000004716.0,9200000005781.5,9900000001721.0
제품명,195,183,콜드 브루,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1회 제공량(kcal),195,80,0,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
포화지방(g),195,37,0,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN
단백질(g),195,17,0,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN
지방(g),195,43,0,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN
트랜스지방(g),195,4,0,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
나트륨(mg),195,48,0,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
당류(g),195,47,0,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("중복데이터 확인")

# 전체 행 중복 확인
print("전체 행 중복 확인")
print(f"portfolio 중복 행 수: {df_portfolio.duplicated().sum()}")
print(f"profile 중복 행 수: {df_profile.duplicated().sum()}")
print(f"menu 중복 행 수: {df_menu.duplicated().sum()}")
print(f"transcript 중복 행 수: {df_transcript.duplicated().sum()}")

중복데이터 확인
전체 행 중복 확인
portfolio 중복 행 수: 0
profile 중복 행 수: 0
menu 중복 행 수: 0
transcript 중복 행 수: 0


In [8]:
# 결측치 확인
print("결측치 확인")
print(f"portfolio 결측치:\n{df_portfolio.isnull().sum()}\n")
print(f"profile 결측치:\n{df_profile.isnull().sum()}\n")
print(f"menu 결측치:\n{df_menu.isnull().sum()}\n")
print(f"transcript 결측치:\n{df_transcript.isnull().sum()}\n")

결측치 확인
portfolio 결측치:
Unnamed: 0    0
reward        0
channels      0
difficulty    0
duration      0
offer_type    0
id            0
dtype: int64

profile 결측치:
Unnamed: 0             0
gender              2175
age                    0
id                     0
became_member_on       0
income              2175
dtype: int64

menu 결측치:
Unnamed: 0      0
제품코드            0
제품명             0
1회 제공량(kcal)    0
포화지방(g)         0
단백질(g)          0
지방(g)           0
트랜스지방(g)        0
나트륨(mg)         0
당류(g)           0
카페인(mg)         0
콜레스테롤(mg)       0
탄수화물(g)         0
dtype: int64

transcript 결측치:
Unnamed: 0    0
person        0
event         0
value         0
time          0
dtype: int64



In [ ]:
# 결측치 상세 확인
print(df_profile['gender'].value_counts(dropna=False))
print(df_profile['income'].value_counts(dropna=False))

gender
M      8484
F      6129
NaN    2175
O       212
Name: count, dtype: int64
income
NaN         2175
73000.0      314
72000.0      297
71000.0      294
57000.0      288
53000.0      282
74000.0      282
52000.0      281
56000.0      281
54000.0      272
70000.0      270
51000.0      268
64000.0      258
61000.0      258
55000.0      254
50000.0      253
60000.0      251
59000.0      243
75000.0      243
67000.0      242
58000.0      239
65000.0      238
62000.0      229
68000.0      228
31000.0      223
66000.0      222
32000.0      216
33000.0      213
37000.0      209
39000.0      209
63000.0      206
35000.0      206
41000.0      204
34000.0      199
69000.0      196
45000.0      196
40000.0      195
47000.0      192
36000.0      192
48000.0      186
38000.0      185
46000.0      181
42000.0      181
44000.0      180
43000.0      169
79000.0      166
77000.0      158
76000.0      157
49000.0      157
78000.0      157
80000.0      152
87000.0      151
84000.0      148
81000.0    

### 1만불 단위로 구간을 만들었을때 구간별 인기 메뉴와 프로모션 참여율

In [24]:
df_profile['income'].median()


np.float64(64000.0)

In [26]:
df_profile['income'].mean()

np.float64(65404.9915682968)

# 결측치 처리
### 성별은 Unknown 으로 채우기
### 소득은 평균값으로 채우기

In [28]:
# 성별 결측치를 'Unknown'으로 채우기
df_profile['gender'] = df_profile['gender'].fillna('unknown')

# 소득 결측치를 평균값으로 채우기
income_mean = df_profile['income'].mean()
df_profile['income'] = df_profile['income'].fillna(income_mean)

# 확인
print(df_profile.isnull().sum())

Unnamed: 0          0
gender              0
age                 0
id                  0
became_member_on    0
income              0
dtype: int64


In [36]:
# 1. 공통적으로 '}' 와 "'" 를 제거
clean_value = df_transcript['value'].str.replace(r"[\{\}']", "", regex=True)

# 2. ':' 를 기준으로 나누고 마지막 값을 가져옴
# 'offer'가 포함된 행은 offer_id로, 'amount'가 포함된 행은 amount로 나눠줌
df_transcript['offer_id'] = clean_value.str.split(':').str[-1].str.strip()
df_transcript.loc[~df_transcript['value'].str.contains('offer'), 'offer_id'] = None

df_transcript['amount'] = clean_value.str.split(':').str[-1].str.strip()
df_transcript.loc[~df_transcript['value'].str.contains('amount'), 'amount'] = None
df_transcript['amount'] = pd.to_numeric(df_transcript['amount'])

# 결과 확인
print(df_transcript[['event', 'offer_id', 'amount']].tail(10))

                  event offer_id  amount
306524      transaction     None   25.14
306525      transaction     None   43.58
306526      transaction     None   22.64
306527  offer completed        2     NaN
306528      transaction     None    2.20
306529      transaction     None    1.59
306530      transaction     None    9.53
306531      transaction     None    3.61
306532      transaction     None    3.53
306533      transaction     None    4.05


In [37]:
# 통합(Merge) 코드 한 줄 팁
df_merge = df_transcript.merge(df_profile, left_on='person', right_on='id', how='left') \
                        .merge(df_portfolio, left_on='offer_id', right_on='id', how='left')

df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0_x      306534 non-null  int64  
 1   person            306534 non-null  str    
 2   event             306534 non-null  str    
 3   value             306534 non-null  str    
 4   time              306534 non-null  int64  
 5   offer_id          167581 non-null  object 
 6   amount            138953 non-null  float64
 7   Unnamed: 0_y      306534 non-null  int64  
 8   gender            306534 non-null  str    
 9   age               306534 non-null  int64  
 10  id_x              306534 non-null  str    
 11  became_member_on  306534 non-null  int64  
 12  income            306534 non-null  float64
 13  Unnamed: 0        134002 non-null  float64
 14  reward            134002 non-null  float64
 15  channels          134002 non-null  str    
 16  difficulty        134002 non-nu

In [ ]:
# 1. 의미 없는 컬럼 삭제
cols_to_drop = ['Unnamed: 0_x', 'Unnamed: 0_y', 'Unnamed: 0', 'id_x', 'id_y', 'value']
df_merge = df_merge.drop(columns=[c for c in cols_to_drop if c in df_merge.columns])

# 2. 컬럼명 변경
df_merge = df_merge.rename(columns={
    'person': 'customer_id',
    'income': 'customer_income',
    'age': 'customer_age'
})

# 3. 결측치 확인
print(df_merge.isnull().sum())

customer_id              0
event                    0
time                     0
offer_id            138953
amount              167581
gender                   0
customer_age             0
became_member_on         0
customer_income          0
reward              172532
channels            172532
difficulty          172532
duration            172532
offer_type          172532
dtype: int64


In [ ]:
# 수치형 데이터 0으로 채우기 (존재하지 않는 기록이기 때문에 최빈값이나 평균, 중앙값으로 채우지 않음)
num_cols = ['amount', 'reward', 'difficulty', 'duration']
df_merge[num_cols] = df_merge[num_cols].fillna(0)

# 범주형 데이터 채우기
df_merge['offer_id'] = df_merge['offer_id'].fillna('no_offer')
df_merge['offer_type'] = df_merge['offer_type'].fillna('transaction')
df_merge['channels'] = df_merge['channels'].fillna('none')

# 최종 확인
print(df_merge.isnull().sum())

customer_id         0
event               0
time                0
offer_id            0
amount              0
gender              0
customer_age        0
became_member_on    0
customer_income     0
reward              0
channels            0
difficulty          0
duration            0
offer_type          0
dtype: int64


In [42]:
# 데이터 추출
df_merge.to_csv('../../Data/merged_song.csv', index=False)